In [ ]:
!pip install transformer_embeddings

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.5/887.5 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 46.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 55.3 MB/s eta 0:00:00
  Attempting uninstall: torch
    Found existing installation: torch 2.3.0+cu121
    Uninstalling torch-2.3.0+cu121:
      Successfully uninstalled torch-2.3.0+cu121
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.3.0+cu121 requires torch==2.3.0, but you have torch 1.13.1 which is incompatible.
torchtext 0.18.0 requires torch>=2.3.0, but you have torch 1.13.1 which is incompatible.
torchvision 0.18.0+cu121 requires torch==2.3.0, but you have torc

In [ ]:
pip install spellchecker contractions empath readability syntok fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 41.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Using cached pybind11-2.12.0-py3-none-any.whl (234 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 30.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 kB 13.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for spellchecker: filename=spellchecker-0.4-py3-none-any.whl size=3966500 sha256=2388fdf34872901c77b0c41b903e1cea371b42a732bd9e39a32e71fd5c183b84
  Stored in directory: /root/.cache/pip/wheels/6c/90/c3/eac248d8755b2a7343487a2087b4b29ad

In [ ]:
!pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.7/224.7 kB 5.7 MB/s eta 0:00:00


In [ ]:
pip install pyspellchecker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 16.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
from spellchecker import SpellChecker
import contractions
from empath import Empath
import readability
import syntok.segmenter as segmenter
from sentence_transformers import SentenceTransformer
import fasttext.util
import numpy as np
from transformer_embeddings import TransformerEmbeddings, mean_pooling
from huggingface_hub import login

In [ ]:
data = pd.read_csv('/content/Reddit_depression_dataset.csv')

In [ ]:
data

,text,label
0,"He said he had not felt that way before, sugge...",mild
1,"Hey there r/assistance, Not sure if this is th...",minimum
2,My mom then hit me with the newspaper and it s...,minimum
3,"until i met my new boyfriend, he is amazing, h...",mild
4,October is Domestic Violence Awareness Month a...,moderate
...,...,...
3548,i have horrible vivid nightmares every night. ...,moderate
3549,Also I can't think about both of them without ...,moderate
3550,"Furthermore, I told him before we got really s...",mild
3551,Here's the link to my amazon wish list where t...,minimum


In [ ]:
data.iloc[3549]['text']

"Also I can't think about both of them without getting angry and jealous again, but to talk to each of them is actually helping me a lot.   ---   **tl;dr**: Three Friends where one M was fallen in Love with F and F was fallen in Love with the other M, which isn't feeling anything for her as an Girlfriend. We could be the best template for an 2,5h long Teenager-Drama.  ---"

In [ ]:
data['text']

0       He said he had not felt that way before, sugge...
1       Hey there r/assistance, Not sure if this is th...
2       My mom then hit me with the newspaper and it s...
3       until i met my new boyfriend, he is amazing, h...
4       October is Domestic Violence Awareness Month a...
                              ...                        
3548    i have horrible vivid nightmares every night. ...
3549    Also I can't think about both of them without ...
3550    Furthermore, I told him before we got really s...
3551    Here's the link to my amazon wish list where t...
3552    How can I keep us protected? They have already...
Name: text, Length: 3553, dtype: object

In [ ]:
def tokenize(text):
    return '\n\n'.join(
     '\n'.join(' '.join(token.value for token in sentence)
        for sentence in paragraph)
     for paragraph in segmenter.analyze(text))

In [ ]:
def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_html(text):
    html_pattern = re.compile('<.*?>')
    return html_pattern.sub(r'', text)

data['text'] = data['text'].apply(lambda x: remove_urls(x))
data['text'] = data['text'].apply(lambda x: remove_html(x))

chat_words_str = """
AFAIK=As Far As I Know
AFK=Away From Keyboard
ASAP=As Soon As Possible
ATK=At The Keyboard
ATM=At The Moment
A3=Anytime, Anywhere, Anyplace
BAK=Back At Keyboard
BBL=Be Back Later
BBS=Be Back Soon
BFN=Bye For Now
B4N=Bye For Now
BRB=Be Right Back
BRT=Be Right There
BTW=By The Way
B4=Before
B4N=Bye For Now
B.S.=bullshit
B.S.,=bullshit,
BS=bullshit
BFF=Best Friends Forever
CU=See You
CUL8R=See You Later
CYA=See You
FAQ=Frequently Asked Questions
FC=Fingers Crossed
FWIW=For What It's Worth
FYI=For Your Information
GAL=Get A Life
GG=Good Game
GN=Good Night
GMTA=Great Minds Think Alike
GR8=Great!
G9=Genius
IC=I See
ICQ=I Seek you (also a chat program)
ILU=ILU: I Love You
IMHO=In My Honest/Humble Opinion
IMO=In My Opinion
IOW=In Other Words
IRL=In Real Life
KISS=Keep It Simple, Stupid
LDR=Long Distance Relationship
LMAO=Laugh My A.. Off
LOL=Laughing Out Loud
LTNS=Long Time No See
L8R=Later
MTE=My Thoughts Exactly
M8=Mate
NRN=No Reply Necessary
OIC=Oh I See
PITA=Pain In The A..
PRT=Party
PRW=Parents Are Watching
ROFL=Rolling On The Floor Laughing
ROFLOL=Rolling On The Floor Laughing Out Loud
ROTFLMAO=Rolling On The Floor Laughing My A.. Off
SK8=Skate
STATS=Your sex and age
ASL=Age, Sex, Location
THX=Thank You
TTFN=Ta-Ta For Now!
TTYL=Talk To You Later
U=You
U2=You Too
U4E=Yours For Ever
WB=Welcome Back
WTF=What The F...
WTG=Way To Go!
WUF=Where Are You From?
W8=Wait...
7K=Sick:-D Laugher
"""

chat_words_map_dict = {}
chat_words_list = []
for line in chat_words_str.split("\n"):
    if line != "":
        cw = line.split("=")[0]
        cw_expanded = line.split("=")[1]
        chat_words_list.append(cw)
        chat_words_map_dict[cw] = cw_expanded
chat_words_list = set(chat_words_list)

def chat_words_conversion(text):
    new_text = []
    for w in text.split():
        if w.upper() in chat_words_list:
            new_text.append(chat_words_map_dict[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text)

data['text'] = data['text'].apply(lambda x: chat_words_conversion(x))

# creating an empty list
def expand_contractions(text):
    expanded_words = []
    for word in text.split():
        # using contractions.fix to expand the shortened words
        expanded_words.append(contractions.fix(word))

    expanded_text = ' '.join(expanded_words)
    return expanded_text

data['text'] = data['text'].apply(lambda x: expand_contractions(x))

spell = SpellChecker()
def correct_spellings(text):
    corrected_text = []
    misspelled_words = spell.unknown(text.split())
    for word in text.split():
        if word in misspelled_words:
            corrected_word = spell.correction(word)
            if corrected_word != None:
                corrected_text.append(corrected_word)
            else:
                corrected_text.append(word)
        else:
            corrected_text.append(word)
    try:
        return " ".join(corrected_text)
    except:
        print(text)

data['text'] = data['text'].apply(lambda x: correct_spellings(x))

In [ ]:
data['text'].iloc[0]

"He said he had not felt that way before suggested I go rest and so ..TRIGGER AHEAD IF YOUI'RE A HYPOCONDRIAC LIKE ME: i decide to look up feelings of doom in hopes of maybe getting sucked into some rabbit hole of ludicrous conspiracy a stupid are you psychic test or new age bullshit something I could even laugh at down the road No, I ended up reading that this sense of doom can be indicative of various health ailments one of which I am prone to So on top of my doom to my gloom..I am now fun worried about my heart I do happen to have a physical in 48 hours"

In [ ]:
data

,text,label
0,He said he had not felt that way before sugges...,mild
1,"Hey there r/assistance, Not sure if this is th...",minimum
2,My mom then hit me with the newspaper and it s...,minimum
3,until i met my new boyfriend he is amazing he ...,mild
4,October is Domestic Violence Awareness Month a...,moderate
...,...,...
3548,i have horrible vivid nightmares every night s...,moderate
3549,Also I cannot think about both of them without...,moderate
3550,"Furthermore, I told him before we got really s...",mild
3551,Here is the link to my amazon wish list where ...,minimum


In [ ]:
for index, row in data.iterrows():
    f = open('/content/txt/{}.txt'.format(index), 'w')
    f.write(row['text'])
    f.close()

In [ ]:
import shutil

shutil.make_archive('txt', 'zip', 'txt')

'/content/txt.zip'

In [ ]:
from google.colab import files

files.download('txt.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
data['label'] = data['label'].map({'minimum': 0, 'mild': 1, 'moderate': 2, 'severe' : 3})

In [ ]:
data

,text,label
0,He said he had not felt that way before sugges...,1
1,"Hey there r/assistance, Not sure if this is th...",0
2,My mom then hit me with the newspaper and it s...,0
3,until i met my new boyfriend he is amazing he ...,1
4,October is Domestic Violence Awareness Month a...,2
...,...,...
3548,i have horrible vivid nightmares every night s...,2
3549,Also I cannot think about both of them without...,2
3550,"Furthermore, I told him before we got really s...",1
3551,Here is the link to my amazon wish list where ...,0


In [ ]:
data_seance = pd.read_csv('/content/results.csv')

In [ ]:
data_seance['id'] = data_seance['filename'].apply(lambda x: x.split('.')[0])
data_seance = data_seance.set_index('id').drop('filename', axis=1)
data.index.name = 'id'
data_seance.index = data_seance.index.astype(int)

In [ ]:
data = data.join(data_seance, how='inner')

In [ ]:
data

,text,label,nwords,Admiration/Awe_GALC,Amusement_GALC,Anger_GALC,Anxiety_GALC,Beingtouched_GALC,Boredom_GALC,Compassion_GALC,...,virtue_adverbs_component,positive_nouns_component,respect_component,trust_verbs_component,failure_component,well_being_component,economy_component,certainty_component,positive_verbs_component,objects_component
id,,,,,,,,,,,,,,,,,,,,,
0,He said he had not felt that way before sugges...,1,113,0.0,0.017699,0.000000,0.008850,0.000000,0.0,0.0,...,0.444753,-0.635348,0.177684,0.214367,0.017053,0.308501,0.289359,0.218807,-0.872000,0.034399
1,"Hey there r/assistance, Not sure if this is th...",0,111,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.005730,0.863369,0.084032,0.171506,0.015429,0.000000,0.021383,0.114919,0.906135,0.180283
2,My mom then hit me with the newspaper and it s...,0,167,0.0,0.005988,0.000000,0.000000,0.017964,0.0,0.0,...,0.000000,0.000000,0.000000,0.066249,0.000000,0.074509,0.522192,0.115689,0.011461,0.156417
3,until i met my new boyfriend he is amazing he ...,1,283,0.0,0.000000,0.003534,0.000000,0.003534,0.0,0.0,...,0.173659,0.581206,0.196990,0.149123,0.023879,0.121663,0.345603,0.179596,0.310530,0.117580
4,October is Domestic Violence Awareness Month a...,2,90,0.0,0.000000,0.022222,0.000000,0.000000,0.0,0.0,...,0.368876,-0.000233,0.010058,0.339390,0.078603,0.295835,0.597996,0.155633,-0.277722,0.099162
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3548,i have horrible vivid nightmares every night s...,2,56,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,-0.879357,0.000000,0.139173,0.087654,0.201198,0.011000,0.230732,-0.872000,0.122610
3549,Also I cannot think about both of them without...,2,72,0.0,0.000000,0.013889,0.000000,0.000000,0.0,0.0,...,0.000000,0.876639,0.000000,0.306850,0.000000,0.010389,0.000000,0.189851,-0.275653,0.008361
3550,"Furthermore, I told him before we got really s...",1,75,0.0,0.000000,0.000000,0.013333,0.000000,0.0,0.0,...,0.008480,-0.875213,0.000000,0.050262,0.112262,0.475667,0.000000,0.087613,-0.872000,0.257464


In [ ]:
lexicon = Empath()

def get_empath_features(post):
    return lexicon.analyze(post, normalize=True)

data['empath_features'] = data['text'].apply(get_empath_features)

data = pd.concat([data, pd.DataFrame(list(data.empath_features)).add_prefix('emp_')], axis=1)
del data['empath_features']

In [ ]:
data

,text,label,nwords,Admiration/Awe_GALC,Amusement_GALC,Anger_GALC,Anxiety_GALC,Beingtouched_GALC,Boredom_GALC,Compassion_GALC,...,emp_weapon,emp_children,emp_monster,emp_ocean,emp_giving,emp_contentment,emp_writing,emp_rural,emp_positive_emotion,emp_musical
0,He said he had not felt that way before sugges...,1,113,0.0,0.017699,0.000000,0.008850,0.000000,0.0,0.0,...,0.0,0.008850,0.0,0.0,0.000000,0.000000,0.008850,0.008850,0.000000,0.000000
1,"Hey there r/assistance, Not sure if this is th...",0,111,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.009009,0.000000,0.000000,0.000000,0.000000,0.000000
2,My mom then hit me with the newspaper and it s...,0,167,0.0,0.005988,0.000000,0.000000,0.017964,0.0,0.0,...,0.0,0.005988,0.0,0.0,0.000000,0.000000,0.005988,0.000000,0.029940,0.005988
3,until i met my new boyfriend he is amazing he ...,1,283,0.0,0.000000,0.003534,0.000000,0.003534,0.0,0.0,...,0.0,0.021201,0.0,0.0,0.000000,0.035336,0.000000,0.000000,0.017668,0.000000
4,October is Domestic Violence Awareness Month a...,2,90,0.0,0.000000,0.022222,0.000000,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.011111,0.011111,0.000000,0.000000,0.011111,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3548,i have horrible vivid nightmares every night s...,2,56,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3549,Also I cannot think about both of them without...,2,72,0.0,0.000000,0.013889,0.000000,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.000000,0.013889,0.000000,0.000000,0.013889,0.000000
3550,"Furthermore, I told him before we got really s...",1,75,0.0,0.000000,0.000000,0.013333,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3551,Here is the link to my amazon wish list where ...,0,64,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.015625,0.0,0.0,0.015625,0.000000,0.015625,0.000000,0.031250,0.000000


In [ ]:
data.to_csv('/content/almostpreprocessed/data_preliwc.csv')

In [ ]:
data['tokenized'] = data['text'].apply(tokenize)
od = data.apply(lambda x: readability.getmeasures(x['tokenized'], lang='en', merge=True), axis=1)
od = od.apply(pd.Series).add_prefix('readability_')
data = pd.concat([data, od], axis=1).drop(['tokenized'], axis=1)

In [ ]:
data.columns

Index(['text', 'label', 'nwords', 'Admiration/Awe_GALC', 'Amusement_GALC',
       'Anger_GALC', 'Anxiety_GALC', 'Beingtouched_GALC', 'Boredom_GALC',
       'Compassion_GALC',
       ...
       'readability_complex_words_dc', 'readability_tobeverb',
       'readability_auxverb', 'readability_conjunction', 'readability_pronoun',
       'readability_preposition', 'readability_nominalization',
       'readability_interrogative', 'readability_article',
       'readability_subordination'],
      dtype='object', length=499)

Embeddings

In [ ]:
model = SentenceTransformer('all-mpnet-base-v2')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = model.encode(data.text, show_progress_bar=True)
data['transformer_embeddings'] = embeddings.tolist()

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

In [ ]:
fasttext.util.download_model('en', if_exists='ignore')  # English

ValueError: ../pretrained-models/cc.en.300.bin cannot be opened for loading!

In [ ]:
ft = fasttext.load_model('/content/cc.en.300.bin')

In [ ]:
# create a new column called embeddings and apply the function to the dataset
data['ft_embeddings'] = data['text'].apply(lambda x: ft.get_sentence_vector(x).tolist())

In [ ]:
data.to_csv('/content/preprocessedData.csv', index=False)

In [ ]:
data.columns

Index(['text', 'label', 'nwords', 'Admiration/Awe_GALC', 'Amusement_GALC',
       'Anger_GALC', 'Anxiety_GALC', 'Beingtouched_GALC', 'Boredom_GALC',
       'Compassion_GALC',
       ...
       'readability_auxverb', 'readability_conjunction', 'readability_pronoun',
       'readability_preposition', 'readability_nominalization',
       'readability_interrogative', 'readability_article',
       'readability_subordination', 'transformer_embeddings', 'ft_embeddings'],
      dtype='object', length=501)

In [ ]:
model = SentenceTransformer('all-distilroberta-v1')
embeddings = model.encode(data.text, show_progress_bar=True)
data['distil_roberta_embeddings'] = embeddings.tolist()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

In [ ]:
data.to_csv('/content/preprocessedData.csv', index=False)

In [ ]:
data

,text,label,nwords,Admiration/Awe_GALC,Amusement_GALC,Anger_GALC,Anxiety_GALC,Beingtouched_GALC,Boredom_GALC,Compassion_GALC,...,readability_conjunction,readability_pronoun,readability_preposition,readability_nominalization,readability_interrogative,readability_article,readability_subordination,transformer_embeddings,ft_embeddings,distil_roberta_embeddings
0,He said he had not felt that way before sugges...,1,113,0.0,0.017699,0.000000,0.008850,0.000000,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,"[-0.0006743494886904955, -0.01285542268306017,...","[-0.007061576936393976, -0.013467751443386078,...","[-0.061126187443733215, 0.007638456299901009, ..."
1,"Hey there r/assistance, Not sure if this is th...",0,111,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,"[0.022605326026678085, 0.015060145407915115, -...","[-0.007316493894904852, 0.0044315471313893795,...","[0.003977155778557062, 0.0017879154765978456, ..."
2,My mom then hit me with the newspaper and it s...,0,167,0.0,0.005988,0.000000,0.000000,0.017964,0.0,0.0,...,0.0,1.0,0.0,3.0,0.0,0.0,0.0,"[0.07074099034070969, -0.003647521836683154, -...","[-0.00012301577953621745, 0.010379962623119354...","[-0.015357895754277706, 0.04006315767765045, -..."
3,until i met my new boyfriend he is amazing he ...,1,283,0.0,0.000000,0.003534,0.000000,0.003534,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,"[0.014903290197253227, 0.021377546712756157, 0...","[0.0045671225525438786, -0.009834645316004753,...","[-0.012589546851813793, 0.014668954536318779, ..."
4,October is Domestic Violence Awareness Month a...,2,90,0.0,0.000000,0.022222,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,"[0.03318581357598305, 0.07784625142812729, 0.0...","[-0.016323337331414223, -0.009591460227966309,...","[0.011043553240597248, -0.037799008190631866, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3548,i have horrible vivid nightmares every night s...,2,56,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,"[0.04299173131585121, -0.004474240355193615, -...","[-0.014431629329919815, -0.0011266920482739806...","[0.004387537017464638, -0.04855622723698616, 0..."
3549,Also I cannot think about both of them without...,2,72,0.0,0.000000,0.013889,0.000000,0.000000,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,"[0.03265056014060974, -0.013125469908118248, -...","[0.00039376740460284054, -0.014767365530133247...","[0.001753989141434431, -0.020550284534692764, ..."
3550,"Furthermore, I told him before we got really s...",1,75,0.0,0.000000,0.000000,0.013333,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[0.025660289451479912, -0.01998322829604149, -...","[-0.0038584074936807156, -0.015921691432595253...","[-0.04715738818049431, 0.06682712584733963, 0...."
3551,Here is the link to my amazon wish list where ...,0,64,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,"[0.01763133332133293, 0.01317767333239317, -0....","[-0.0008206831989809871, -0.00781549047678709,...","[-0.009727059863507748, 0.000536331266630441, ..."
